In [ ]:
import keras
import tensorflow as tf
from time import strftime
from pathlib import Path
from os import mkdir

def lr_schedule(epoch, lr):
    base_lr=0.01
    if epoch < 2:
        return base_lr
    elif epoch < 4:
        return base_lr * 0.5
    elif epoch < 7: 
        return base_lr * 0.05
    elif epoch < 11:
        return base_lr * 0.005
    elif epoch < 15:
        return base_lr * 0.0005
    else:
        return base_lr * 0.00005

def get_run_log_dir(root_dir='my_logs'):
    root_path=Path(root_dir)
    root_path.mkdir(exist_ok=True,parents=True)
    return Path(root_dir)/strftime("run_%Y_%m_%d_%H_%M_%S")

run_dir=get_run_log_dir()

cifar=keras.datasets.cifar10.load_data()

(X_train,y_train),(X_test,y_test)=cifar

X_train,X_test=X_train/255.0,X_test/255.0

X_valid,y_valid=X_train[-5000:],y_train[-5000:]

X_train,y_train=X_train[:-5000],y_train[:-5000]

tf.random.set_seed(0)

elu_act=keras.activations.elu

he_init=keras.initializers.HeNormal()

l2_reg=keras.regularizers.l2()

data_aug=keras.Sequential([
    keras.layers.RandomFlip(mode='horizontal'),
    keras.layers.RandomTranslation(0.1,0.1),
])

inputs=keras.layers.Input(shape=(32,32,3),name='input')

x=data_aug(inputs)

x=keras.layers.Conv2D(32,(3,3),padding='same',kernel_regularizer=l2_reg, kernel_initializer=he_init)(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(activation=elu_act)(x)
x=keras.layers.Dropout(0.2)(x)
x=keras.layers.Conv2D(32,(3,3),padding='same',kernel_regularizer=l2_reg, kernel_initializer=he_init)(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(activation=elu_act)(x)
x=keras.layers.Dropout(0.2)(x)
block_1_output=keras.layers.MaxPool2D(pool_size=(2,2))(x)

x=keras.layers.Conv2D(64,(3,3),padding='same',kernel_regularizer=l2_reg, kernel_initializer=he_init)(block_1_output)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(activation=elu_act)(x)
x=keras.layers.Dropout(0.2)(x)
x=keras.layers.Conv2D(64,(3,3),padding='same',kernel_regularizer=l2_reg, kernel_initializer=he_init)(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(activation=elu_act)(x)
x=keras.layers.Dropout(0.2)(x)
skip_1=keras.layers.Conv2D(64,(1,1),padding='same',kernel_regularizer=l2_reg, kernel_initializer=he_init)(block_1_output)
x=keras.layers.add([x,skip_1])
x=keras.layers.MaxPool2D(2,2)(x)
block_2_output=x

x=keras.layers.Conv2D(128,(3,3),padding='same',kernel_regularizer=l2_reg, kernel_initializer=he_init)(block_2_output)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(activation=elu_act)(x)
x=keras.layers.Dropout(0.2)(x)
x=keras.layers.Conv2D(128,(3,3),padding='same',kernel_regularizer=l2_reg, kernel_initializer=he_init)(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(activation=elu_act)(x)
x=keras.layers.Dropout(0.2)(x)
skip_2=keras.layers.Conv2D(128,(1,1),padding='same',kernel_regularizer=l2_reg, kernel_initializer=he_init)(block_2_output)
x=keras.layers.add([x,skip_2])
x=keras.layers.MaxPool2D(2,2)(x)
block_3_output=x

flatten=keras.layers.GlobalMaxPooling2D()(block_3_output)
x=keras.layers.Dense(128, activation=elu_act, kernel_initializer=he_init, kernel_regularizer=l2_reg)(flatten)
outputs=keras.layers.Dense(10, activation='softmax')(x)

model=keras.Model(inputs=inputs, outputs=outputs)

SGD_optimizer=keras.optimizers.SGD(learning_rate=0.01, nesterov=True, momentum=0.9)

Adam_optimizer=keras.optimizers.Adam(learning_rate=0.0001, clipnorm=1.0)

loss=keras.losses.SparseCategoricalCrossentropy()

model.compile(optimizer=SGD_optimizer,loss=loss,metrics=['accuracy'])

tensorboard_cb=keras.callbacks.TensorBoard(run_dir)

earlyStop_cb=keras.callbacks.EarlyStopping(monitor='val_accuracy',patience=7,verbose=1, restore_best_weights=True)

lrPlateau_cb=keras.callbacks.ReduceLROnPlateau(monitor='val_accuracy',factor=0.05, patience=3,verbose=1, min_lr=1e-6)

LrScedule_cb=keras.callbacks.LearningRateScheduler(lr_schedule, verbose=1)

history=model.fit(
    X_train,y_train,
    epochs=40,
    verbose=1,
    validation_data=(X_valid,y_valid),
    batch_size=32,
    callbacks=[tensorboard_cb ,lrPlateau_cb, earlyStop_cb]
)

model.save("Saved Models/Residual_CNN_3x3_3ConvBlocks_ELU_SGD_BN_DO02_LearningRatePlateau_FlipSwitch.keras")

2026-01-18 15:31:00.051865: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M2
2026-01-18 15:31:00.051914: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 8.00 GB
2026-01-18 15:31:00.051918: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 2.67 GB
2026-01-18 15:31:00.051939: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-01-18 15:31:00.052260: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


Epoch 1/40


2026-01-18 15:31:02.680135: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


1407/1407 ━━━━━━━━━━━━━━━━━━━━ 91s 62ms/step - accuracy: 0.4068 - loss: 4.8420 - val_accuracy: 0.4612 - val_loss: 1.9261 - learning_rate: 0.0100
Epoch 2/40
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 89s 63ms/step - accuracy: 0.5380 - loss: 1.7824 - val_accuracy: 0.2934 - val_loss: 2.6797 - learning_rate: 0.0100
Epoch 3/40
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 90s 64ms/step - accuracy: 0.5672 - loss: 1.7275 - val_accuracy: 0.3478 - val_loss: 2.5066 - learning_rate: 0.0100
Epoch 4/40
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step - accuracy: 0.5804 - loss: 1.7141
Epoch 4: ReduceLROnPlateau reducing learning rate to 0.0004999999888241291.
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 90s 64ms/step - accuracy: 0.5849 - loss: 1.7054 - val_accuracy: 0.4206 - val_loss: 2.3107 - learning_rate: 0.0100
Epoch 5/40
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 91s 65ms/step - accuracy: 0.6738 - loss: 1.4310 - val_accuracy: 0.6964 - val_loss: 1.3442 - learning_rate: 5.0000e-04
Epoch 6/40
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 91s 65ms/step - accuracy: 0.714